In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame named 'forrest'
train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')

# Display the first few rows of the DataFrame to verify the data has been loaded
print(train.head())

In [ ]:
train.info()

In [ ]:
#check if there is any class imbalancing
train['Cover_Type'].value_counts(normalize= True)

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

scaler = StandardScaler()

# Extract features and target variable
X = train.drop(['Id', 'Cover_Type'], axis=1)
y = train['Cover_Type']

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
import numpy as np

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Subtract 1 from class labels to convert them to start from 0
y_train_encoded = to_categorical(y_train - 1)
y_val_encoded = to_categorical(y_val - 1)

# Scale the data using the previously fitted scaler
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)  # Use transform, not fit_transform

# Define hyperparameters for XGBoost
param_grid_xgboost = {
    'n_estimators': [50,100, 150],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [7],
    'min_child_weight': [1],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

# Create XGBoost model
xgb_model = XGBClassifier()

# Perform GridSearchCV for XGBoost
grid_xgboost = GridSearchCV(estimator=xgb_model, param_grid=param_grid_xgboost, cv=5, scoring='accuracy', n_jobs=-1, verbose=True)
grid_result_xgboost = grid_xgboost.fit(X_train_scaled, y_train_encoded)

# Print the best parameters and corresponding accuracy
print("Best Parameters for XGBoost: ", grid_result_xgboost.best_params_)
print("Best Accuracy for XGBoost: ", grid_result_xgboost.best_score_)

# Make predictions on the validation set using the best model from grid search
best_xgb_model = grid_result_xgboost.best_estimator_
y_pred = best_xgb_model.predict(X_val_scaled)

# Evaluate accuracy for this fold
accuracy = accuracy_score(y_val_encoded, y_pred)
print("Accuracy on Validation Set: ", accuracy)


**Submission**

In [ ]:
import pandas as pd

# Load the test set into a DataFrame
test_set = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

# Display the first few rows of the test set
print(test_set.head())

In [ ]:
X_test = test_set.drop(['Id'], axis=1)
# Apply the same scaling to the test set features
X_test_scaled = scaler.transform(X_test.values)

# Make predictions
predictions = best_xgb_model.predict(X_test_scaled)

# Assuming you have a classification task and want to get the predicted class labels
predicted_labels = np.argmax(predictions, axis=1) + 1  # Add 1 to convert back to class labels starting from 1

# Create a DataFrame with the observation Id and predicted cover type
submission_df = pd.DataFrame({'Id': test_set['Id'], 'Cover_Type': predicted_labels})

print(submission_df)

# Save the DataFrame to a CSV file
submission_df.to_csv('submission_file.csv', index=False)